In [9]:
!pip install -q langchain==0.2.10
!pip install -q langchain-openai==0.1.17
!pip install -q langchain-groq==0.1.6
!pip install rich

In [14]:
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import FewShotChatMessagePromptTemplate

from rich.console import Console
from rich.markdown import Markdown


In [15]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=userdata.get("GROQ_API_KEY")
)


console = Console()


In [29]:
planet_name = "Neptune"
adjective = "Funny"

In [25]:
messages = [
    ("system", f"You are a alien robot from {planet_name}\nAnswer shortly"),
    ("human", "how cold are the nights?")
]

In [26]:
response = llm.invoke(messages)

In [27]:
console.print(Markdown(response.content))

On Neptune, night temps plunge to roughly ‑200 °C (about ‑300 °F).

## Instead of f-strings, we can leverage prompts!

In [30]:
prompt_template = PromptTemplate.from_template("Tell me a {adjective} fact about {planet_name}")


prompt = prompt_template.format(adjective=adjective, planet_name=planet_name)

resp = llm.invoke(prompt)

console.print(Markdown(resp.content))

Sure! Here’s a goofy nugget about our distant blue giant:                                                          

Neptune’s “weather forecast” is basically a nonstop, planet‑wide fireworks show.                                   

 • The planet’s atmosphere is a roiling sea of hydrogen, helium, and methane, and it hosts the fastest winds in the
   Solar System, whipping around at up to 2,100 km/h (about 1,300 mph)—fast enough to circle the globe in just a   
   few Earth days.                                                                                                 
 • Those supersonic jet streams constantly stir up massive, dark “storm spots” (the most famous being the Great    
   Dark Spot, a cousin of Jupiter’s Great Red Spot).                                                               
 • And because the winds are so extreme, they can lift clouds of methane ice crystals into towering, glittering    
   “cloud‑spires” that look like the planet is wearing a glittery, ever‑changing crown.                            

So, if you ever imagined Neptune as a calm, icy blue marble, think again—its atmosphere is basically the universe’s
most elaborate, high‑speed, glitter‑filled rave, and the party never stops! 🎉🌊🚀

## We can also have chat templates

In [38]:
major_name = "cars"
adjective = "funny"
planet_name = "Mars"

In [39]:
chat_template = ChatPromptTemplate.from_messages(
    [("system", ("You are an expert in {major}")),
     ("human", ("Answer with {adjective} style")),
     ("ai", "sure!"),
     ("human", "tell me a fact about {planet_name}")]
)


chat = chat_template.format_messages(major=major_name, adjective=adjective, planet_name=planet_name)

response = llm.invoke(chat)

In [40]:
console.print(Markdown(response.content))

🌌 Mars Fact (with a side of giggles):                                                                             

Mars is home to the tallest mountain in the solar system—Olympus Mons. It’s about 22 kilometers (13.6 mi) high,    
which is roughly three times the height of Mount Everest.                                                          

So if you ever feel like your morning commute is a mountain of traffic, just remember: even the Red Planet’s       
biggest hill would make your rush‑hour jam look like a gentle stroll up a curb! 🚗💨                               

(Bonus: if you’re planning a vacation there, just pack extra oxygen, a sturdy pair of hiking boots, and maybe a    
really, really long ladder.)

## Few-shot Prompts

In [46]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

In [55]:
examples = [
    {"input":"hi", "output":"سلااااااام!"},
    {"input":"bye!", "output":"فعلااااااااا!"}
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"), ("ai", "{output}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(example_prompt=example_prompt, examples=examples)

prompt = ChatPromptTemplate([
    ("system", "You are a English-Farsi translator"),
    few_shot_prompt,
    ("human", "{input}")
])

In [56]:
chain = prompt | llm

In [57]:
resp = chain.invoke({"input":"How are you?"})

In [58]:
console.print(Markdown(resp.content))

English: How are you?                                                                                              
Farsi (Persian): حال شما چطور است؟ / چطوری؟

## We can have our answers in json format

In [59]:
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers.json import SimpleJsonOutputParser


In [63]:
json_prompt = PromptTemplate.from_template("Return a JSON object with an 'answer' key that answers the following question: {question}")
json_parser = SimpleJsonOutputParser()


In [73]:
json_chain = json_prompt | llm | json_parser

In [74]:
resp = json_chain.invoke({"question":"What is the tallest building?"})

In [75]:
print(resp)

{'answer': 'The tallest building in the world is the Burj Khalifa in Dubai, United Arab Emirates, which stands at 828 meters (2,717 feet) tall.'}


In [76]:
console.print(Markdown(resp["answer"]))

The tallest building in the world is the Burj Khalifa in Dubai, United Arab Emirates, which stands at 828 meters   
(2,717 feet) tall.

## We can also use pydantic

In [78]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field

In [102]:
class Code(BaseModel):
  line_by_line_desc: str=Field(description="explain the code line by line")
  python_code: str=Field(description="provide the python code with no comments")


parser = JsonOutputParser(pydantic_object=Code)

In [103]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"line_by_line_desc": {"title": "Line By Line Desc", "description": "explain the code line by line", "type": "string"}, "python_code": {"title": "Python Code", "description": "provide the python code with no comments", "type": "string"}}, "required": ["line_by_line_desc", "python_code"]}
```


In [104]:
prompt = PromptTemplate(
    template="{format_instruction}\nWrite me a function to do {query}",
    input_variables=["query"],
    partial_variables={"format_instruction":parser.get_format_instructions()}
)

In [105]:
chain = prompt | llm | parser

In [106]:
resp = chain.invoke({"query":"write me function to add many numbers"})

In [107]:
print(resp["line_by_line_desc"])

Line 1: Define a function `add_numbers` that accepts a variable number of arguments using `*numbers`.
Line 2: Initialize a variable `total` to 0 to accumulate the sum.
Line 3: Iterate over each number `n` in the provided `numbers`.
Line 4: Add the current number `n` to `total`.
Line 5: After the loop ends, return `total` as the result.


In [108]:
print(resp["python_code"])

def add_numbers(*numbers):
    total = 0
    for n in numbers:
        total += n
    return total
